# EarthSafe — Exploratory Data Analysis
USGS Earthquake Hazards Program | 2023–2024 | M≥2.5

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

DATA_PATH = Path('../data/raw/earthquakes.csv')
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

## 1. Dataset Overview

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Data Types ===')
print(df.dtypes)
print('\n=== Basic Statistics ===')
df[['magnitude', 'depth', 'gap', 'rms']].describe()

## 2. Target Distribution (Hazard Labels)

In [ ]:
label_order = ['Low', 'Moderate', 'High']
colors = {'Low': '#2ecc71', 'Moderate': '#f39c12', 'High': '#e74c3c'}

counts = df['hazard_label'].value_counts()[label_order]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar(counts.index, counts.values,
                   color=[colors[l] for l in counts.index], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title('Hazard Label Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Hazard Level')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(counts.values, labels=counts.index,
            colors=[colors[l] for l in counts.index],
            autopct='%1.1f%%', startangle=140, pctdistance=0.75)
axes[1].set_title('Hazard Label Proportions', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/fig_label_distribution.png', bbox_inches='tight')
plt.show()
print(counts)

## 3. Feature Distributions by Hazard Level

In [ ]:
features = ['magnitude', 'depth', 'gap', 'rms']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, feat in zip(axes, features):
    for label in label_order:
        subset = df[df['hazard_label'] == label][feat].dropna()
        ax.hist(subset, bins=40, alpha=0.6, label=label,
                color=colors[label], density=True, edgecolor='none')
    ax.set_title(f'{feat.capitalize()} Distribution', fontsize=12, fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Feature Distributions by Hazard Level', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/fig_feature_distributions.png', bbox_inches='tight')
plt.show()

## 4. Magnitude vs Depth Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for label in label_order:
    subset = df[df['hazard_label'] == label]
    ax.scatter(subset['magnitude'], subset['depth'],
               alpha=0.4, s=15, label=label, color=colors[label])

ax.set_xlabel('Magnitude', fontsize=12)
ax.set_ylabel('Depth (km)', fontsize=12)
ax.set_title('Magnitude vs Depth by Hazard Level', fontsize=13, fontweight='bold')
ax.axvline(4.0, color='orange', linestyle='--', linewidth=1.5, alpha=0.7, label='Low/Moderate boundary')
ax.axvline(6.0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Moderate/High boundary')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/fig_magnitude_depth_scatter.png', bbox_inches='tight')
plt.show()

## 5. Correlation Heatmap

In [ ]:
numeric_feats = ['magnitude', 'depth', 'gap', 'rms']
corr = df[numeric_feats].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, ax=ax, square=True, linewidths=0.5,
            vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/fig_correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 6. Location Type Breakdown

In [ ]:
loc_hazard = df.groupby(['location_type', 'hazard_label']).size().unstack(fill_value=0)[label_order]
loc_hazard_pct = loc_hazard.div(loc_hazard.sum(axis=1), axis=0) * 100

loc_hazard_pct.plot(kind='bar', stacked=True,
                    color=[colors[l] for l in label_order],
                    figsize=(8, 5), edgecolor='white')
plt.title('Hazard Level by Location Type (%)', fontsize=13, fontweight='bold')
plt.xlabel('Location Type')
plt.ylabel('Percentage')
plt.xticks(rotation=0)
plt.legend(title='Hazard Level')
plt.tight_layout()
plt.savefig('../data/processed/fig_location_type.png', bbox_inches='tight')
plt.show()

## 7. Key EDA Takeaways

In [ ]:
print('=== Key Observations ===')
print(f"Total records: {len(df):,}")
print(f"Label distribution:\n{df['hazard_label'].value_counts()}")
print(f"\nMagnitude range: {df['magnitude'].min():.1f} – {df['magnitude'].max():.1f}")
print(f"Depth range: {df['depth'].min():.1f} – {df['depth'].max():.1f} km")
print(f"\nLocation type counts:\n{df['location_type'].value_counts()}")
print(f"\nClass imbalance ratio (Low:Moderate:High):")
v = df['hazard_label'].value_counts()[label_order]
print(f"  {v['Low']} : {v['Moderate']} : {v['High']} → may need class_weight='balanced'")